# The Python Data Model
### Phase 01 

> *"One of the best qualities of Python is its consistency. After working with Python
> for a while, you are able to start making informed, correct guesses about features
> that are new to you."* — Fluent Python

This notebook is a **standalone reference** for Chapter 1.
Read the actual chapter first. Return here to study, practice, and review.

**How to use:**
- Every concept is built in **pure Python first**
- ML/DL connections come **after** the Python foundation is solid
- Every code cell is runnable — re-run anything, break things, experiment

---

**Sections**

| # | Python Foundation | # | ML Extension |
|---|---|---|---|
| 1 | The Data Model — what and why | 10 | namedtuple → Sample, Batch |
| 2 | `namedtuple` — lightweight schema | 11 | Dataset protocol |
| 3 | The container pattern | 12 | Vector → Tensor |
| 4 | What you get for free | 13 | `__call__` → nn.Module |
| 5 | `__contains__` and sorting | 14 | Production TabularDataset |
| 6 | How special methods work | 15 | Context managers in ML |
| 7 | The Vector class — numeric dunders | | |
| 8 | Overview of all special methods | | |
| 9 | Why `len` is not a method | | |



---
# Part 1 — The Python Data Model: What and Why

## The key insight

If you learned another OO language before Python, you found it strange to write
`len(collection)` instead of `collection.len()`.

This apparent oddity is the tip of an iceberg called the **Python data model**.
It describes the API that lets your own objects play well with the most idiomatic
language features.

> Think of the data model as a description of **Python as a framework**.
> It formalizes the interfaces of the building blocks of the language itself —
> sequences, iterators, functions, classes, context managers, and so on.

When you code with any framework, you spend time implementing methods that are
called by the framework. The same happens with the Python data model:
the interpreter invokes **special methods** to perform basic object operations,
triggered by syntax. These are always written with leading and trailing double
underscores: `__getitem__`, `__len__`, etc. — called **dunder methods**.

**What dunders enable:**
- Iteration
- Collections
- Attribute access
- Operator overloading
- Function and method invocation
- Object creation and destruction
- String representation and formatting
- Managed contexts (`with` blocks)


In [ ]:
# The oddity that unlocks everything:
# Why len(x) and not x.len() ?

my_list = [1, 2, 3, 4, 5]

# Every other OO language would have you write:
# my_list.length()   or   my_list.size()   or   my_list.count()

# Python has ONE consisten t interface for ALL collections:
print(len(my_list))          # list
print(len("hello"))          # str
print(len({1: 'a', 2: 'b'})) # dict
print(len((1, 2, 3)))        # tuple

# Same function, any collection — that's the data model in action.
# When you implement __len__ in your class, len() works on it too.
# One grammar, everything speaks it.


5
5
2
3



---
# Part 2 — `namedtuple`: Lightweight Data Schema

## `namedtuple` — a bundle of attributes with no custom methods

`collections.namedtuple` builds classes that are just bundles of attributes,
like a database record. No custom methods, minimal overhead.

The book uses it to represent individual cards — think of it as defining
**the schema of one item** in a collection.


In [93]:
import collections

# Book: a playing card — two fields, immutable, named
Card = collections.namedtuple('Card', ['rank', 'suit'])

beer_card = Card('7', 'diamonds')
print(beer_card)           # Card(rank='7', suit='diamonds')
print(beer_card.rank)      # '7'  — named field access
print(beer_card.suit)      # 'diamonds'
print(beer_card[0])        # '7'  — still a tuple underneath
print(beer_card[1])        # 'diamonds'

# Unpackable — like a tuple
rank, suit = beer_card
print(rank, suit)

# Immutable — cannot be changed
try:
    beer_card.rank = '8'
except AttributeError as e:
    print(f"Immutable: {e}")


Card(rank='7', suit='diamonds')
7
diamonds
7
diamonds
7 diamonds
Immutable: can't set attribute


### `namedtuple` vs `dict` vs `class`

| Feature | `dict` | `class` | `namedtuple` |
|---|---|---|---|
| Named field access | ✓ | ✓ | ✓ |
| Immutable by default | ✗ | ✗ | ✓ |
| Memory efficient | ✗ | ✗ | ✓ |
| Indexable / unpackable | ✗ | ✗ | ✓ |
| Minimal boilerplate | ✓ | ✗ | ✓ |
| Readable `repr` | ✓ | ✗ | ✓ |
| Hashable by default | ✗ | ✗ | ✓ |
| Supports methods | ✗ | ✓ | Limited |

**Rule:** use `namedtuple` when data is mostly **read-only structured records**.


---
# Part 3 — The Container Pattern: `__len__` + `__getitem__`

## FrenchDeck — the book's central example

A short class that packs a punch. By implementing just two special methods —
`__len__` and `__getitem__` — it behaves like a full standard Python sequence.

The cards are irrelevant. **The pattern is everything.**


In [ ]:
import collections
from random import choice

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]


deck = FrenchDeck()
print(f"Total cards: {len(deck)}")
print(f"First card : {deck[0]}")
print(f"Last card  : {deck[-1]}")


Total cards: 52
First card : Card(rank='2', suit='spades')
Last card  : Card(rank='A', suit='hearts')



---
# Part 4 — What You Get For Free

## The payoff of speaking Python's grammar

By implementing `__len__` and `__getitem__`, `FrenchDeck` gets all of
this **without a single additional line of implementation**:


In [95]:
# ── 1. Indexing ──────────────────────────────────────────────────
print("=== Indexing ===")
print(deck[0])     # first
print(deck[-1])    # last — negative indexing free

# ── 2. Slicing ───────────────────────────────────────────────────
print("\n=== Slicing ===")
print(deck[:3])          # first three
print(deck[12::13])      # aces — every 13th card starting from index 12

# ── 3. random.choice ─────────────────────────────────────────────
print("\n=== random.choice ===")
print(choice(deck))
print(choice(deck))

# ── 4. Iteration ─────────────────────────────────────────────────
print("\n=== Iteration (first 3) ===")
for i, card in enumerate(deck):
    print(card)
    if i == 2: break

# ── 5. Reversed iteration ────────────────────────────────────────
print("\n=== Reversed (first 3) ===")
for i, card in enumerate(reversed(deck)):
    print(card)
    if i == 2: break


=== Indexing ===
Card(rank='2', suit='spades')
Card(rank='A', suit='hearts')

=== Slicing ===
[Card(rank='2', suit='spades'), Card(rank='3', suit='spades'), Card(rank='4', suit='spades')]
[Card(rank='A', suit='spades'), Card(rank='A', suit='diamonds'), Card(rank='A', suit='clubs'), Card(rank='A', suit='hearts')]

=== random.choice ===
Card(rank='J', suit='spades')
Card(rank='A', suit='clubs')

=== Iteration (first 3) ===
Card(rank='2', suit='spades')
Card(rank='3', suit='spades')
Card(rank='4', suit='spades')

=== Reversed (first 3) ===
Card(rank='A', suit='hearts')
Card(rank='K', suit='hearts')
Card(rank='Q', suit='hearts')


### Why does `random.choice` work without implementing it?

```python
# random.choice internally does roughly this:
def choice(seq):
    return seq[int(random() * len(seq))]   # needs __len__ + __getitem__
```

It only needs two questions answered: *"how many?"* and *"give me item N"*.
You answered both. That's the contract.

**Two advantages the book highlights:**

1. Users don't memorise arbitrary method names — is it `.size()`, `.length()`,
   `.count()`? With dunders there's one answer: `len()`.

2. You benefit from the rich Python standard library without reinventing it —
   `random.choice`, `sorted`, `reversed`, `in` — all free.



---
# Part 5 — `__contains__` and Sorting

## `in` operator — sequential scan vs `__contains__`

If a class has **no `__contains__` method**, the `in` operator does a
**sequential scan** — it iterates through every item until it finds a match.

`FrenchDeck` has no `__contains__`, but it is iterable (via `__getitem__`),
so `in` works — just not in O(1).


In [ ]:
# __contains__ not defined — in does a sequential scan via iteration
print(Card('Q', 'hearts') in deck)   # True  — found it
print(Card('7', 'beasts') in deck)   # False — 'beasts' is not a suit

# This works BUT it scans up to 52 cards.
# For a 52-card deck that's fine. For a million-record dataset — not fine.
# Phase 02 will show how __contains__ on a set/dict gives O(1).


True
False


## Sorting with a key function

`FrenchDeck` doesn't define `__lt__` or any comparison dunders.
But `sorted()` only needs iteration — which we have via `__getitem__`.
Pass a `key` function and sorted works with any iterable:


In [97]:
# Rank cards: spades=3 (highest), hearts=2, diamonds=1, clubs=0 (lowest)
suit_values = dict(spades=3, hearts=2, diamonds=1, clubs=0)

def spades_high(card):
    rank_value = FrenchDeck.ranks.index(card.rank)
    return rank_value * len(suit_values) + suit_values[card.suit]

# Sort the entire deck from lowest (2 of clubs) to highest (A of spades)
sorted_deck = sorted(deck, key=spades_high)
print("Lowest 3:", sorted_deck[:3])
print("Highest 3:", sorted_deck[-3:])

# sorted() uses __iter__ → __getitem__ under the hood.
# The key function maps each item to a comparable value.
# This pattern is everywhere in ML: sort sequences by length,
# sort samples by label, rank predictions by confidence score.


Lowest 3: [Card(rank='2', suit='clubs'), Card(rank='2', suit='diamonds'), Card(rank='2', suit='hearts')]
Highest 3: [Card(rank='A', suit='diamonds'), Card(rank='A', suit='hearts'), Card(rank='A', suit='spades')]


## Why can't we shuffle?

`FrenchDeck` is currently **immutable at the sequence level** — its positions
cannot be reassigned. `random.shuffle` needs to swap items in place, which
requires `__setitem__`:

```python
random.shuffle(deck)
# TypeError: 'FrenchDeck' object does not support item assignment
```

The fix is one line — adding `__setitem__`:
```python
def __setitem__(self, position, value):
    self._cards[position] = value
```

We'll implement this properly in Phase 05 (Ch 12).


In [98]:
import random

# Demonstrate the limitation:
try:
    random.shuffle(deck)
except TypeError as e:
    print(f"Can't shuffle: {e}")

# Fix — add __setitem__:
class MutableDeck(FrenchDeck):
    def __setitem__(self, position, value):
        self._cards[position] = value

mutable = MutableDeck()
random.shuffle(mutable)
print("Shuffled first 5:", mutable[:5])


Can't shuffle: 'FrenchDeck' object does not support item assignment
Shuffled first 5: [Card(rank='9', suit='clubs'), Card(rank='A', suit='clubs'), Card(rank='K', suit='hearts'), Card(rank='2', suit='clubs'), Card(rank='K', suit='diamonds')]



---
# Part 6 — How Special Methods Are Actually Used

## The three rules

**Rule 1:** Special methods are meant to be called by the **Python interpreter**,
not by you.

**Rule 2:** For built-in types (`list`, `str`, `dict`), the interpreter takes a
shortcut — it reads directly from a C struct field, bypassing Python entirely.

**Rule 3:** Special method calls are mostly **implicit** — triggered by syntax,
not by explicit calls.


In [99]:
# Rule 1 — Never call dunders directly
class MyCollection:
    def __init__(self, data): self._data = data
    def __len__(self): return len(self._data)

c = MyCollection([1, 2, 3])

# Wrong — bypasses interpreter protections:
c.__len__()    # works but wrong

# Right — goes through the built-in:
len(c)         # correct

# The difference matters: len() enforces a contract
class BrokenCollection:
    def __len__(self): return -1  # nonsense

bc = BrokenCollection()
try:
    len(bc)               # raises ValueError
except ValueError as e:
    print(f"len() caught it: {e}")

print(bc.__len__())       # returns -1 silently — no protection


len() caught it: __len__() should return >= 0
-1


## Why `len()` on built-ins is extremely fast in CPython

For many built-in variable-sized objects (`list`, `tuple`, `str`, `bytes`, etc.),
CPython stores the object's size directly inside the object header.

At the C level, many objects inherit from `PyVarObject`:

```c
typedef struct {
    Py_ssize_t ob_refcnt;   // reference count
    PyTypeObject *ob_type; // pointer to the object's type
    Py_ssize_t ob_size;    // number of contained elements
} PyVarObject;
````

For these built-ins, `len(obj)` can often return the size with a direct
C-level memory read from `ob_size`.

That means:

* no Python attribute lookup
* no Python bytecode execution
* no dynamic method resolution

just a tiny O(1) C operation.


## Mental Model

Built-in containers:

```text
len(obj)
    ↓
read cached size from object memory
```

Custom Python objects:

```text
len(obj)
    ↓
dispatch to __len__()
    ↓
execute Python-defined logic
```

Both are fast.

But built-ins are specially optimized because their size is stored directly in the CPython object layout.

```
```


In [1]:
import timeit

data = list(range(10_000))


class WrappedList:

    def __init__(self, data):
        self._data = data

    def __len__(self):

        # This inner len() still uses the optimized
        # built-in C implementation for list objects.
        return len(self._data)


wrapped = WrappedList(data)

t_list = timeit.timeit(
    lambda: len(data),
    number=1_000_000
)

t_wrapped = timeit.timeit(
    lambda: len(wrapped),
    number=1_000_000
)

print(f"len(list)   : {t_list:.4f}s")
print(f"len(custom) : {t_wrapped:.4f}s")
print(f"Ratio       : {t_wrapped / t_list:.1f}x slower")

print()
print("Built-in containers store their size directly in the")
print("CPython object layout, so len(list) is nearly a raw")
print("C-level memory read.")

print()
print("Custom classes require dispatch to the object's")
print("__len__ implementation before returning a result.")

print()
print("Important:")
print("The overhead is NOT computing the length itself.")
print("Inside __len__, len(self._data) still uses the")
print("optimized built-in C fast path.")

len(list)   : 0.1007s
len(custom) : 0.2286s
Ratio       : 2.3x slower

Built-in containers store their size directly in the
CPython object layout, so len(list) is nearly a raw
C-level memory read.

Custom classes require dispatch to the object's
__len__ implementation before returning a result.

Important:
The overhead is NOT computing the length itself.
Inside __len__, len(self._data) still uses the
optimized built-in C fast path.


## The complete implicit dunder map

Every Python operator, statement, and built-in is secretly a dunder call:

| Syntax / Built-in | Dunder called |
|---|---|
| `len(x)` | `x.__len__()` |
| `x[i]` | `x.__getitem__(i)` |
| `x[i] = v` | `x.__setitem__(i, v)` |
| `del x[i]` | `x.__delitem__(i)` |
| `for i in x:` | `iter(x)` → `x.__iter__()` |
| `x in collection` | `collection.__contains__(x)` → sequential scan |
| `x + y` | `x.__add__(y)` |
| `x * n` | `x.__mul__(n)` |
| `n * x` | `x.__rmul__(n)` (reversed) |
| `abs(x)` | `x.__abs__()` |
| `str(x)` | `x.__str__()` |
| `repr(x)` | `x.__repr__()` |
| `if x:` | `x.__bool__()` → fallback `x.__len__()` |
| `with x:` | `x.__enter__()` / `x.__exit__()` |
| `x(args)` | `x.__call__(args)` |
| `x == y` | `x.__eq__(y)` |
| `x < y` | `x.__lt__(y)` |
| `hash(x)` | `x.__hash__()` |

**`__init__` is the only dunder you call directly** — when chaining to a
parent class via `super().__init__()`. Everything else goes through the built-in.


In [101]:
# Demonstrate implicit calls — each line triggers a different dunder

class Verbose:
    """A class that announces every dunder call it receives."""
    def __init__(self, data):
        self._data = list(data)

    def __len__(self):
        print("  → __len__ called")
        return len(self._data)

    def __getitem__(self, i):
        print(f"  → __getitem__({i}) called")
        return self._data[i]

    def __contains__(self, item):
        print(f"  → __contains__({item}) called")
        return item in self._data

    def __repr__(self):
        return f"Verbose({self._data})"

    def __bool__(self):
        print("  → __bool__ called")
        return len(self._data) > 0

v = Verbose([10, 20, 30])

print("len(v):")
_ = len(v)

print("\nv[0]:")
_ = v[0]

print("\n20 in v:")
_ = 20 in v

print("\nif v:")
if v: pass

print("\nfor x in v (first item only):")
for x in v:
    print(f"  got {x}")
    break


len(v):
  → __len__ called

v[0]:
  → __getitem__(0) called

20 in v:
  → __contains__(20) called

if v:
  → __bool__ called

for x in v (first item only):
  → __getitem__(0) called
  got 10


## One important rule: don't invent `__foo__` names

```python
class Model:
    def __train__(self): ...   # dangerous — never do this
```

Python reserves the `__name__` namespace for itself. A name that's meaningless
today may become a real protocol in a future Python version.
Your code silently breaks. Use plain names for your own methods.



---
# Part 7 — Emulating Numeric Types: The Vector Class

## The book's second example — a 2D Vector

The `FrenchDeck` showed the **sequence** side of the data model.
Now the `Vector` class shows the **numeric** side.

We will implement: `__repr__`, `__abs__`, `__bool__`, `__add__`, `__mul__`.

The design goal — make this work naturally:
```python
v1 = Vector(2, 4)
v2 = Vector(2, 1)
v1 + v2        # → Vector(4, 5)
abs(v1)        # → 4.47...  (Euclidean magnitude)
v1 * 3         # → Vector(6, 12)
bool(Vector(0, 0))  # → False  (zero vector is falsy)
```


In [ ]:
from math import hypot

class Vector:
    """
    Two-dimensional vector — exact book implementation.
    Demonstrates: __repr__, __abs__, __bool__, __add__, __mul__
    """
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r})'

    def __abs__(self):
        return hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))   # False if magnitude is 0

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Vector(x, y)      # returns a NEW Vector — never modifies self

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)


v1 = Vector(2, 4)
v2 = Vector(2, 1)

print(repr(v1))        # __repr__  → Vector(2, 4)
print(v1 + v2)         # __add__   → Vector(4, 5)
print(abs(v1))         # __abs__   → 4.47...
print(v1 * 3)          # __mul__   → Vector(6, 12)
print(abs(v1 * 3))     # → 13.41...
print(bool(v1))        # __bool__  → True
print(bool(Vector(0, 0)))  # → False


Vector(2, 4)
Vector(4, 5)
4.47213595499958
Vector(6, 12)
13.416407864998739
True
False


## The immutability principle — infix operators return new objects

This is explicitly stated in the book and is **critical for ML**:

> *"Note that `__add__` and `__mul__` create and return a new instance of Vector,
> and do not modify either operand — `self` or `other` are merely read.
> This is the expected behavior of infix operators: to create new objects
> and not touch their operands."*

**Why this matters in ML:** PyTorch tensors follow this exact principle.
`t1 + t2` never modifies `t1` or `t2` — it creates a new tensor.
This is what allows safe gradient computation in autograd.


In [103]:
# Prove immutability — infix operators never modify operands
v1 = Vector(2, 4)
v2 = Vector(2, 1)

v3 = v1 + v2       # creates a new Vector

print(f"v1 unchanged: {v1}")   # still Vector(2, 4)
print(f"v2 unchanged: {v2}")   # still Vector(2, 1)
print(f"v3 is new   : {v3}")   # Vector(4, 5)
print(f"v3 is v1    : {v3 is v1}")  # False — different object


v1 unchanged: Vector(2, 4)
v2 unchanged: Vector(2, 1)
v3 is new   : Vector(4, 5)
v3 is v1    : False


## `__abs__` — magnitude as a built-in

`abs()` is a built-in, just like `len()`.
For numbers: absolute value. For complex numbers: magnitude.
For our Vector: Euclidean magnitude = `√(x² + y²)`.

This is `hypot(x, y)` — the same formula as the L2 norm in ML.

```python
abs(Vector(3, 4))  # → 5.0  (3-4-5 right triangle)
```

By implementing `__abs__`, our Vector integrates with `abs()` just like
`complex` numbers do — consistent, no new API to learn.


In [104]:
# __abs__ — Euclidean magnitude
print(abs(Vector(3, 4)))    # 5.0  — classic 3-4-5 triangle
print(abs(Vector(0, 0)))    # 0.0  — zero vector
print(abs(Vector(1, 0)))    # 1.0  — unit vector

# abs() result feeds directly into __bool__:
# bool(abs(Vector(0,0))) = bool(0.0) = False
# bool(abs(Vector(3,4))) = bool(5.0) = True


5.0
0.0
1.0


## `__bool__` — the zero vector is falsy

The book's implementation: `return bool(abs(self))`

There's a faster alternative that avoids the square root:
```python
def __bool__(self):
    return bool(self.x or self.y)
```

`self.x or self.y` short-circuits — if `x` is nonzero, truthy immediately.
Only if `x` is zero does it check `y`. No `abs()`, no `hypot()`, no square root.


In [105]:
class VectorFast(Vector):
    def __bool__(self):
        return bool(self.x or self.y)   # avoids hypot entirely

# Same behavior, lower cost:
print(bool(VectorFast(0, 0)))    # False
print(bool(VectorFast(1, 0)))    # True
print(bool(VectorFast(0, 1)))    # True
print(bool(VectorFast(3, 4)))    # True

import timeit
v_zero = Vector(0, 0)
vf_zero = VectorFast(0, 0)
t1 = timeit.timeit(lambda: bool(v_zero),  number=1_000_000)
t2 = timeit.timeit(lambda: bool(vf_zero), number=1_000_000)
print(f"\nbool with abs()     : {t1:.4f}s")
print(f"bool with x or y    : {t2:.4f}s")
print(f"Speedup             : {t1/t2:.1f}x")


False
True
True
True

bool with abs()     : 0.4931s
bool with x or y    : 0.2067s
Speedup             : 2.4x


## `__rmul__` — why `3 * v` fails without it

Our `__mul__` handles `v * 3` (Vector on the left).
But `3 * v` (scalar on the left) calls `int.__mul__(3, v)` first —
which doesn't know about Vector and returns `NotImplemented`.
Python then tries `v.__rmul__(3)` — the **reversed** version.

Without `__rmul__`, `3 * v` raises `TypeError`.


In [106]:
v = Vector(2, 4)

# This works — Vector on the left:
print(v * 3)           # __mul__(3) → Vector(6, 12)

# This fails — scalar on the left, no __rmul__:
try:
    print(3 * v)
except TypeError as e:
    print(f"TypeError: {e}")

# Fix — add __rmul__:
class VectorFull(Vector):
    def __rmul__(self, scalar):
        return self * scalar   # delegate to __mul__

vf = VectorFull(2, 4)
print(3 * vf)          # now works → VectorFull(6, 12)
print(vf * 3)          # still works → VectorFull(6, 12)


Vector(6, 12)
TypeError: unsupported operand type(s) for *: 'int' and 'Vector'
Vector(6, 12)
Vector(6, 12)



---
# Part 8 — Overview of All Special Methods

## The complete map — 83 special methods

The Python Language Reference defines 83 special methods.
47 are for arithmetic, bitwise, and comparison operators.

### Table 1 — Special methods (operators excluded)

| Category | Methods |
|---|---|
| String/bytes representation | `__repr__`, `__str__`, `__format__`, `__bytes__` |
| Conversion to number | `__abs__`, `__bool__`, `__complex__`, `__int__`, `__float__`, `__hash__`, `__index__` |
| Emulating collections | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__` |
| Iteration | `__iter__`, `__reversed__`, `__next__` |
| Emulating callables | `__call__` |
| Context management | `__enter__`, `__exit__` |
| Instance creation and destruction | `__new__`, `__init__`, `__del__` |
| Attribute management | `__getattr__`, `__getattribute__`, `__setattr__`, `__delattr__`, `__dir__` |
| Attribute descriptors | `__get__`, `__set__`, `__delete__` |
| Class services | `__prepare__`, `__instancecheck__`, `__subclasscheck__` |

### Table 2 — Special methods for operators

| Category | Methods |
|---|---|
| Unary numeric | `__neg__` `-`, `__pos__` `+`, `__abs__` `abs()` |
| Rich comparison | `__lt__` `<`, `__le__` `<=`, `__eq__` `==`, `__ne__` `!=`, `__gt__` `>`, `__ge__` `>=` |
| Arithmetic | `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__divmod__`, `__pow__` |
| Reversed arithmetic | `__radd__`, `__rsub__`, `__rmul__`, `__rtruediv__`, `__rfloordiv__`, `__rmod__`, `__rpow__` |
| Augmented assignment | `__iadd__`, `__isub__`, `__imul__`, `__itruediv__`, `__ifloordiv__`, `__imod__`, `__ipow__` |
| Bitwise | `__invert__`, `__lshift__`, `__rshift__`, `__and__`, `__or__`, `__xor__` |

**Reversed operators** are fallbacks used when operands are swapped (`b * a` instead of `a * b`).
**Augmented assignment** operators combine an infix operator with variable assignment (`a += b` → `a.__iadd__(b)`).


In [107]:
# Demonstrate augmented assignment vs regular arithmetic
class Counter:
    def __init__(self, n): self.n = n
    def __repr__(self): return f"Counter({self.n})"

    def __add__(self, other):
        print("  __add__ called — returns new object")
        return Counter(self.n + other.n)

    def __iadd__(self, other):
        print("  __iadd__ called — modifies in place")
        self.n += other.n
        return self       # must return self

a = Counter(10)
b = Counter(5)

print("a + b:")
c = a + b           # __add__ — new object, a unchanged
print(f"  c={c}, a={a}  (a unchanged)")

print("\na += b:")
a += b              # __iadd__ — modifies a in place
print(f"  a={a}  (a modified)")

# Note: if __iadd__ is not defined, += falls back to __add__ + rebinding.
# PyTorch tensors use this distinction: t1 + t2 is new, t1 += t2 is in-place.


a + b:
  __add__ called — returns new object
  c=Counter(15), a=Counter(10)  (a unchanged)

a += b:
  __iadd__ called — modifies in place
  a=Counter(15)  (a modified)



---
# Part 9 — Why `len` Is Not a Method

## The philosophy — practicality beats purity

The book quotes a 2013 answer from core developer Raymond Hettinger:

> *"practicality beats purity"* — The Zen of Python

`len` is not called as a method because it gets **special treatment as part
of the Python data model**, just like `abs`.

For built-in objects of CPython, no method is called at all — the length
is simply read from a field in a C struct. Getting the number of items in
a collection is a common operation and must work efficiently for such basic
and diverse types as `str`, `list`, `memoryview`, and so on.

But thanks to `__len__`, you can also make `len()` work with your own
custom objects. This is a **fair compromise**:
- Built-in types get maximum speed
- User types get consistency with the language

Also from the Zen of Python:
> *"Special cases aren't special enough to break the rules."*

If you think of `abs` and `len` as **unary operators**, you may be more
inclined to forgive their functional look-and-feel, as opposed to the
method call syntax one might expect in an OO language.


In [108]:
# The compromise in action:
import sys

# Built-in: C struct read — essentially free
my_list = list(range(1000))
print(f"len(list)      : {len(my_list)}")

# Custom: Python method dispatch — still fast, consistent interface
class Vocabulary:
    def __init__(self, tokens): self._tokens = set(tokens)
    def __len__(self): return len(self._tokens)

vocab = Vocabulary(['the', 'a', 'is', 'in', 'it'])
print(f"len(Vocabulary): {len(vocab)}")

# Both use the same syntax. That's the whole point.
# You, the user, never think about whether len() is
# reading a C struct or calling a Python method.
# The interface is consistent. Practicality beats purity.


len(list)      : 1000
len(Vocabulary): 5



---
# Part 10 — ML Extension: `namedtuple` → Sample, Batch, MetricResult

## From Card to Sample — the same pattern, real stakes

Now that the Python foundation is solid, the ML connections are obvious.
`namedtuple` is not a card trick — it's a schema tool.


In [109]:
import collections
import numpy as np

# ── Training sample ──────────────────────────────────────────────
Sample = collections.namedtuple('Sample', ['image', 'label'])

s = Sample(image=np.zeros((3, 224, 224)), label=9)
print(f"label  : {s.label}")          # named access — intent is clear
print(f"shape  : {s.image.shape}")
print(f"s[1]   : {s[1]}")             # index access still works
img, lbl = s                           # unpackable

# ── Batch ─────────────────────────────────────────────────────────
Batch = collections.namedtuple('Batch', ['inputs', 'targets', 'mask'])

batch = Batch(
    inputs=np.zeros((32, 3, 224, 224)),
    targets=np.zeros(32, dtype=int),
    mask=np.ones(32, dtype=bool)
)
print(f"\nbatch.inputs.shape  : {batch.inputs.shape}")
print(f"batch.targets.shape : {batch.targets.shape}")

# ── Experiment result ──────────────────────────────────────────────
MetricResult = collections.namedtuple('MetricResult',
                                      ['epoch', 'train_loss', 'val_loss', 'val_acc'])

result = MetricResult(epoch=5, train_loss=0.142, val_loss=0.198, val_acc=0.923)
print(f"\n{result}")                  # readable repr built-in
print(f"val_acc: {result.val_acc:.1%}")

# Store results — namedtuples are hashable (can go in sets/dict keys)
results = {result}
print(f"hashable: {len(results) == 1}")


label  : 9
shape  : (3, 224, 224)
s[1]   : 9

batch.inputs.shape  : (32, 3, 224, 224)
batch.targets.shape : (32,)

MetricResult(epoch=5, train_loss=0.142, val_loss=0.198, val_acc=0.923)
val_acc: 92.3%
hashable: True



---
# Part 11 — ML Extension: FrenchDeck → PyTorch Dataset Protocol

## FrenchDeck IS a Dataset

The structure is identical. The only difference is the domain.
PyTorch's `Dataset` contract requires exactly `__len__` + `__getitem__`.


In [ ]:
import collections
import random
from typing import Optional

Sample = collections.namedtuple('Sample', ['features', 'label'])

class TabularDataset:
    """
    Implements PyTorch Dataset protocol via the Python Data Model.
    No imports needed — pure Python Data Model.
    """
    def __init__(self, data: list):
        self._data = data

    # ── Core protocol (required by DataLoader) ───────────────────
    def __len__(self):
        return len(self._data)

    def __getitem__(self, idx):
        return self._data[idx]

    # ── Extra dunders ─────────────────────────────────────────────
    

    def __bool__(self):
        return len(self) > 0

    def __contains__(self, sample):
        return sample in self._data

    def __iter__(self):
        return iter(self._data)

    # ── Utilities ─────────────────────────────────────────────────
    def split(self, ratio=0.8):
        n = int(len(self) * ratio)
        shuffled = random.sample(self._data, len(self._data))
        return TabularDataset(shuffled[:n]), TabularDataset(shuffled[n:])


# ── Demo ──────────────────────────────────────────────────────────
data = [Sample(features=[i, i*2, i*3], label=i % 3) for i in range(120)]
ds = TabularDataset(data)

print(repr(ds))
print(f"len      : {len(ds)}")
print(f"first    : {ds[0]}")
print(f"last     : {ds[-1]}")
print(f"slice    : {ds[:2]}")
print(f"bool     : {bool(ds)}")
print(f"empty    : {bool(TabularDataset([]))}")

# Sorting by label — same pattern as spades_high
sorted_ds = sorted(ds, key=lambda s: s.label)
print(f"\nSorted first 3: {sorted_ds[:3]}")

train, val = ds.split()
print(f"\ntrain={len(train)}, val={len(val)}")
print(f"random sample: {random.choice(ds)}")


TabularDataset(n=120, labels=[0, 1, 2])
len      : 120
first    : Sample(features=[0, 0, 0], label=0)
last     : Sample(features=[119, 238, 357], label=2)
slice    : [Sample(features=[0, 0, 0], label=0), Sample(features=[1, 2, 3], label=1)]
bool     : True
empty    : False

Sorted first 3: [Sample(features=[0, 0, 0], label=0), Sample(features=[3, 6, 9], label=0), Sample(features=[6, 12, 18], label=0)]

train=96, val=24
random sample: Sample(features=[79, 158, 237], label=1)



---
# Part 12 — ML Extension: Vector → Tensor

## Vector IS a Tensor (the pattern, not the implementation)

Every concept from the `Vector` class maps directly to ML tensor operations.


In [111]:
from math import sqrt

class Tensor1D:
    """
    1D tensor — all the dunders from Vector, extended to n dimensions.
    Shows how PyTorch-style tensor operations are built on the data model.
    """
    def __init__(self, data):
        self._data = list(data)

    def __repr__(self):
        return f"Tensor1D({self._data})"

    def __len__(self):
        return len(self._data)

    def __getitem__(self, idx):
        return self._data[idx]

    def __abs__(self):
        """L2 norm — same as Vector magnitude, same as torch.norm"""
        return sqrt(sum(x**2 for x in self._data))

    def __bool__(self):
        """False only if all-zero tensor"""
        return any(x != 0 for x in self._data)

    def __add__(self, other):
        """Element-wise addition — returns new Tensor, never modifies self"""
        assert len(self) == len(other), "Shape mismatch"
        return Tensor1D(a + b for a, b in zip(self._data, other._data))

    def __mul__(self, scalar):
        """Scalar multiplication — returns new Tensor"""
        return Tensor1D(x * scalar for x in self._data)

    def __rmul__(self, scalar):
        """Handles: 3 * tensor (scalar on the left)"""
        return self * scalar

    def __eq__(self, other):
        return self._data == other._data

    def __neg__(self):
        """Unary negation — same as -tensor in PyTorch"""
        return Tensor1D(-x for x in self._data)


t1 = Tensor1D([1.0, 2.0, 3.0])
t2 = Tensor1D([4.0, 5.0, 6.0])

print(repr(t1))
print(f"t1 + t2    : {t1 + t2}")        # __add__
print(f"t1 * 2     : {t1 * 2}")         # __mul__
print(f"3 * t1     : {3 * t1}")         # __rmul__
print(f"-t1        : {-t1}")            # __neg__
print(f"abs(t1)    : {abs(t1):.4f}")    # __abs__ = L2 norm
print(f"bool(t1)   : {bool(t1)}")       # __bool__
print(f"bool(zeros): {bool(Tensor1D([0, 0, 0]))}")

# Immutability — infix ops never modify operands
t3 = t1 + t2
print(f"\nt1 after t1+t2: {t1}  (unchanged)")
print(f"t3 is new     : {t3 is not t1}")


Tensor1D([1.0, 2.0, 3.0])
t1 + t2    : Tensor1D([5.0, 7.0, 9.0])
t1 * 2     : Tensor1D([2.0, 4.0, 6.0])
3 * t1     : Tensor1D([3.0, 6.0, 9.0])
-t1        : Tensor1D([-1.0, -2.0, -3.0])
abs(t1)    : 3.7417
bool(t1)   : True
bool(zeros): False

t1 after t1+t2: Tensor1D([1.0, 2.0, 3.0])  (unchanged)
t3 is new     : True



---
# Part 13 — ML Extension: `__call__` → nn.Module

## Why `model(x)` works — `__call__` in action

When you write `model(x)` in PyTorch, you're not calling `forward()` directly.
You're triggering `model.__call__(x)`, which PyTorch uses to run hooks,
handle gradients, and then call `forward()`.

This is the **callable** part of the data model.


In [112]:
class Module:
    """Minimal nn.Module — shows __call__ → forward() pattern."""

    def __call__(self, *args, **kwargs):
        """model(x) triggers this, which calls forward(x)."""
        # In real nn.Module: run hooks, handle grad mode, then forward
        return self.forward(*args, **kwargs)

    def forward(self, x):
        raise NotImplementedError

    def __repr__(self):
        return f"{self.__class__.__name__}()"


class Linear(Module):
    def __init__(self, in_f, out_f):
        self.in_f  = in_f
        self.out_f = out_f
        # weights initialized to zero for simplicity
        self.W = [[0.0] * in_f for _ in range(out_f)]
        self.b = [0.0] * out_f

    def forward(self, x):
        return [sum(w*xi for w, xi in zip(row, x)) + bias
                for row, bias in zip(self.W, self.b)]

    def __repr__(self):
        return f"Linear(in_features={self.in_f}, out_features={self.out_f})"


class ReLU(Module):
    def forward(self, x):
        return [max(0, xi) for xi in x]

    def __repr__(self):
        return "ReLU()"


layer  = Linear(4, 3)
relu   = ReLU()
x      = [1.0, -2.0, 3.0, -4.0]

print(repr(layer))
print(repr(relu))

out = layer(x)           # __call__ → forward(x) — NOT layer.forward(x)
print(f"\nlayer(x)  : {out}")
print(f"relu(out) : {relu(out)}")    # callable too


Linear(in_features=4, out_features=3)
ReLU()

layer(x)  : [0.0, 0.0, 0.0]
relu(out) : [0, 0, 0]



---
# Part 14 — ML Extension: `__repr__` for Experiment Configs

## `__repr__` — the most underused tool in ML code

Without `__repr__`, every config object prints as
`<__main__.ModelConfig object at 0x7f3b2c1a4d90>`.
That's useless in logs, notebooks, and debuggers.

The rule from the book: **always implement `__repr__`**.
It should be **unambiguous** and ideally match the constructor call.


In [113]:
class ModelConfig:
    def __init__(self, lr, batch_size, epochs, hidden_dim, dropout=0.1):
        self.lr         = lr
        self.batch_size = batch_size
        self.epochs     = epochs
        self.hidden_dim = hidden_dim
        self.dropout    = dropout

    def __repr__(self):
        # Unambiguous — matches constructor call exactly
        return (f"ModelConfig(lr={self.lr}, batch_size={self.batch_size}, "
                f"epochs={self.epochs}, hidden_dim={self.hidden_dim}, "
                f"dropout={self.dropout})")

    def __str__(self):
        # Human-readable — for display/logging
        return f"[lr={self.lr} | bs={self.batch_size} | epochs={self.epochs}]"

    def __eq__(self, other):
        return (isinstance(other, ModelConfig) and
                self.__dict__ == other.__dict__)

    def __hash__(self):
        # Makes configs usable as dict keys / set members
        return hash((self.lr, self.batch_size, self.epochs,
                     self.hidden_dim, self.dropout))


cfg1 = ModelConfig(lr=1e-3, batch_size=32, epochs=10, hidden_dim=256)
cfg2 = ModelConfig(lr=1e-3, batch_size=32, epochs=10, hidden_dim=256)
cfg3 = ModelConfig(lr=1e-4, batch_size=64, epochs=20, hidden_dim=512)

print(repr(cfg1))    # __repr__ — use in logs
print(str(cfg1))     # __str__  — use in UI
print(cfg1 == cfg2)  # __eq__   → True
print(cfg1 == cfg3)  # __eq__   → False

# Hashable — can track configs in a set (experiment deduplication)
ran = {cfg1, cfg2, cfg3}
print(f"Unique configs: {len(ran)}")  # 2 — cfg1 and cfg2 are equal


ModelConfig(lr=0.001, batch_size=32, epochs=10, hidden_dim=256, dropout=0.1)
[lr=0.001 | bs=32 | epochs=10]
True
False
Unique configs: 2



---
# Part 15 — ML Extension: `__enter__`/`__exit__` → Context Managers

## `with` blocks — `__enter__` and `__exit__`

The `with` statement calls `__enter__` on entry and `__exit__` on exit
(even if an exception is raised). This is the **managed context** part
of the data model — one of the most powerful patterns in production ML.


In [114]:
import time

class Timer:
    """Context manager for timing code blocks."""
    def __enter__(self):
        self.start = time.perf_counter()
        return self                     # available as 'as' target

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"  Elapsed: {self.elapsed*1000:.2f}ms")
        return False                    # don't suppress exceptions

class NoGrad:
    """Minimal version of torch.no_grad() — disables gradient tracking."""
    def __init__(self):
        self._prev = None

    def __enter__(self):
        # In real PyTorch: disables autograd
        print("  [no_grad] gradient tracking disabled")
        return self

    def __exit__(self, *args):
        # In real PyTorch: restores previous grad mode
        print("  [no_grad] gradient tracking restored")
        return False


# Timing a training step:
with Timer() as t:
    total = sum(i**2 for i in range(100_000))

print(f"Result: {total}")

# Inference without gradients:
with NoGrad():
    # model(x) would run here without building the computation graph
    prediction = [0.1, 0.7, 0.2]  # fake logits
    print(f"  Prediction: {prediction}")

# These are exactly how torch.no_grad(), torch.cuda.amp.autocast(),
# and experiment trackers (MLflow, W&B) are implemented.


  Elapsed: 9.73ms
Result: 333328333350000
  [no_grad] gradient tracking disabled
  Prediction: [0.1, 0.7, 0.2]
  [no_grad] gradient tracking restored



---
# Summary — The Python Data Model: One-Page Reference

## The single rule

> **You implement dunders. Python calls them. Built-ins are the interface between the two.**

---

## The minimum viable dataset (PyTorch protocol)
```python
def __len__(self)           → len(), random.choice(), DataLoader batching
def __getitem__(self, idx)  → indexing, slicing, iteration, membership scan
```

## The Vector/Tensor dunder set
```python
def __repr__(self)          → repr(), shell display, logging
def __abs__(self)           → abs() → L2 norm
def __bool__(self)          → if tensor: → falsy if zero/empty
def __add__(self, other)    → t1 + t2 → new object, operands unchanged
def __mul__(self, scalar)   → t * 3
def __rmul__(self, scalar)  → 3 * t (scalar on left)
def __neg__(self)           → -t
def __iadd__(self, other)   → t1 += t2 (in-place, returns self)
```

## The fallback chain
```
if x:  →  __bool__  →  (not defined?)  →  __len__  →  (not defined?)  →  always True
```

## The immutability principle
Infix operators (`+`, `*`, `-`) **return new objects** — never modify operands.
This is what makes autograd safe.

## Never do this
```python
obj.__len__()           # call len(obj)
obj.__repr__()          # call repr(obj)
def __train__(self):    # don't invent dunder names
```

## Production habits
| Habit | Reason |
|---|---|
| Always implement `__repr__` | Useless default: `<object at 0x7f...>` |
| Capitalize namedtuple names: `Card`, `Sample` | Convention — avoids shadowing |
| Use `__bool__` only for custom logic | `__len__` fallback covers "non-empty = True" |
| Implement `__contains__` for large collections | Sequential scan is O(n) |
| Never implement `__foo__` custom names | Reserved for Python |

## ML framework connection map
| Python Data Model | ML Framework |
|---|---|
| `__len__` + `__getitem__` | PyTorch `Dataset` protocol |
| `__call__` | `nn.Module` — `model(x)` → `forward(x)` |
| `__repr__` | Config/layer descriptions in experiment logs |
| `__iter__` | `for batch in dataloader:` |
| `__enter__`/`__exit__` | `torch.no_grad()`, `autocast()`, timers |
| `__add__`/`__mul__` returning new objects | Autograd-safe tensor ops |
| `__abs__` | Tensor norm (`torch.norm`) |
| `__bool__` | `if tensor:` — zero tensor is falsy |

## The bigger picture
`FrenchDeck` inherits from `object` but its **power comes from composition**,
not inheritance. It delegates to a list via `__getitem__` and `__len__`.
This gives it all sequence behaviors without inheriting from any sequence type.

This is **Pythonic design**: implement the right protocol, get the full
language and ecosystem for free.
